In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def read_dataset(path):
    df = pd.read_csv(path, header=None, names=['x','y','label'])
    points = df[['x','y']].to_numpy()
    labels = df['label'].to_numpy()
    return points, labels

def fit_perceptron_heuristic(points, labels, alpha, n_iterations):
    weights = np.random.randn(points.shape[1])
    bias = 0.0

    trajectory = [(weights.copy(), bias)]
    for _ in range(n_iterations):
        for pt, lbl in zip(points, labels):
            activation = weights.dot(pt) + bias
            prediction = 1 if activation > 0 else 0
            error = lbl - prediction
            if error != 0:
                delta = alpha * error
                weights += delta * pt
                bias += delta
        trajectory.append((weights.copy(), bias))

    return trajectory

def compute_decision_line(params, x_vals):
    w, b = params
    if abs(w[1]) < 1e-8:
        return np.full_like(x_vals, -b / w[0])
    return -(w[0] * x_vals + b) / w[1]

def render_boundaries(trajectory, points, labels, title):
    fig = go.Figure()

    for cls, color in zip((0,1), ('royalblue','orangered')):
        mask = (labels == cls)
        fig.add_trace(go.Scatter(
            x=points[mask,0], y=points[mask,1],
            mode='markers',
            marker=dict(color=color, size=8),
            name=f'Class {cls}'
        ))
    x_min, x_max = points[:,0].min() - 0.1, points[:,0].max() + 0.1
    xs = np.array([x_min, x_max])

    for idx, params in enumerate(trajectory):
        ys = compute_decision_line(params, xs)

        if idx == 0:
            line_style = dict(color='red', dash='solid', width=3)
            legend_name, show_legend = 'Initial', True
        elif idx == len(trajectory) - 1:
            line_style = dict(color='black', dash='solid', width=3)
            legend_name, show_legend = 'Final', True
        else:
            line_style = dict(color='green', dash='dash', width=1)
            legend_name, show_legend = None, False

        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode='lines',
            line=line_style,
            name=legend_name,
            showlegend=show_legend
        ))

    fig.update_layout(
        title=title,
        xaxis_title='x₁',
        yaxis_title='x₂',
        xaxis=dict(range=[x_min, x_max]),
        yaxis=dict(range=[points[:,1].min() - 0.1, points[:,1].max() + 0.1])
    )
    fig.show()

def main():
    np.random.seed(123)
    X, y = read_dataset('data.csv')

    for n_iter in (15,20, 40, 80,95):
        for alpha in (0.01, 0.1, 1.0):
            traj = fit_perceptron_heuristic(X, y, alpha, n_iter)
            render_boundaries(
                traj, X, y,
                title=f'Heuristic Perceptron α={alpha}, epochs={n_iter}'
            )

if __name__ == '__main__':
    main()


In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def get_data(filename):
    df = pd.read_csv(filename, header=None, names=['f1','f2','target'])
    features = df[['f1','f2']].values
    labels   = df['target'].values
    return features, labels

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def train_model(features, labels, rate, num_epochs):
    w = np.random.randn(features.shape[1])
    b = 0.0

    boundary_steps = [(w.copy(), b)]
    losses = []

    for epoch in range(num_epochs):
        for x, y in zip(features, labels):
            z = w.dot(x) + b
            pred = sigmoid(z)
            error = y - pred
            w += rate * error * x
            b += rate * error

        boundary_steps.append((w.copy(), b))

        z_all = features.dot(w) + b
        p_all = sigmoid(z_all)
        loss = -np.mean(labels * np.log(p_all + 1e-9) +
                        (1 - labels) * np.log(1 - p_all + 1e-9))
        losses.append(loss)

    return boundary_steps, losses

def display_results(features, labels, rates, epoch_list):
    for epochs in epoch_list:
        cols = len(rates)
        fig = make_subplots(
            rows=2, cols=cols,
            subplot_titles=(
                [f"Boundary (η={r}, E={epochs})" for r in rates] +
                [f"Loss (η={r})"            for r in rates]
            )
        )

        for idx, rate in enumerate(rates):
            boundaries, loss_vals = train_model(features, labels, rate, epochs)
            for cls, clr in zip([0,1], ['navy','crimson']):
                mask = labels == cls
                fig.add_trace(
                    go.Scatter(
                        x=features[mask,0], y=features[mask,1],
                        mode='markers',
                        marker=dict(color=clr, size=7),
                        name=f"Class {cls}",
                        showlegend=(idx == 0)
                    ),
                    row=1, col=idx+1
                )

            x0, x1 = 0.0, 1.0
            for step, (w_step, b_step) in enumerate(boundaries):
                if abs(w_step[1]) > 1e-6:
                    y0 = -(w_step[0]*x0 + b_step)/w_step[1]
                    y1 = -(w_step[0]*x1 + b_step)/w_step[1]
                else:
                    y0, y1 = 0.0, 1.0

                if step == 0:
                    color, style, width, lbl = 'red',   'solid', 3, 'Start'
                elif step == len(boundaries)-1:
                    color, style, width, lbl = 'black', 'solid', 3, 'End'
                else:
                    color, style, width, lbl = 'green', 'dash',  1, None

                fig.add_trace(
                    go.Scatter(
                        x=[x0, x1], y=[y0, y1],
                        mode='lines',
                        line=dict(color=color, dash=style, width=width),
                        name=lbl,
                        showlegend=(lbl is not None and idx==0)
                    ),
                    row=1, col=idx+1
                )

            fig.update_xaxes(range=[0,1], title_text='f₁', row=1, col=idx+1)
            fig.update_yaxes(range=[0,1], title_text='f₂', row=1, col=idx+1)

            fig.add_trace(
                go.Scatter(
                    x=list(range(1, len(loss_vals)+1)),
                    y=loss_vals,
                    mode='lines+markers',
                    showlegend=False
                ),
                row=2, col=idx+1
            )
            fig.update_xaxes(title_text='Epoch',    row=2, col=idx+1)
            fig.update_yaxes(title_text='Log Loss', row=2, col=idx+1)

        fig.update_layout(
            title_text=f"Logistic‑Style Perceptron – {epochs} Epochs",
            height=800, width=1400
        )
        fig.show()

if __name__ == '__main__':
    np.random.seed(0)
    X, y = get_data('data.csv')
    learning_rates = [0.01, 0.1, 1.0]
    epochs_to_try  = [100, 150, 180,200]
    display_results(X, y, learning_rates, epochs_to_try)
